# Task 7: Evaluation with a Kaggle model

**Settings:** create this notebook from **https://www.kaggle.com/benchmarks/tasks/new** (a benchmark notebook with
`kaggle_benchmarks` and model access), then load these cells with File > Import Notebook. Accelerator **None** (CPU),
Internet **on**. **Input:** Add Input > Your Work > the saved output of **pa2-06-generate**.
Run with **Save Version > Save & Run All (Commit)**.

Judging reads the saved pair files only; nothing is regenerated. The judge, `google/gemini-3.6-flash`, sees each pair
only as instruction, Response A, Response B (plus category and expected behaviour for the safety sets), never the model
identities. Four judgements:

| Comparison | Question | Pairs |
|---|---|---|
| `dpo_hh_vs_sft` | overall quality, with the safety rubric and each prompt's expected behaviour | 300 |
| `dpo_ultrafeedback_vs_sft` | overall quality | 300 |
| `lora_vs_base_persona` | which answer is more in the Ramsay persona | 40 |
| `lora_vs_base_persona` | usefulness and safety | 40 |

Reliability (manual section 13.2): 5 pairs per request, at least 5 s before every request, up to 3 attempts per batch, a fresh
chat per request, verdicts checkpointed after every batch, unresolved batches in `*_failures.jsonl`, and reruns resume from the
existing judgements. Code: `src/pa2/judge.py` (also `scripts/judge_pairwise_kaggle.py`). Results are archived as
`task7_judge_outputs.zip`.

In [ ]:
REPO_URL = "https://github.com/mabdullahi7780/small-model-alignment-starter.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working"
REPO_DIR = "/tmp/small-model-alignment-starter"
OUT_DIR = f"{WORK_DIR}/outputs/judge"
JUDGE_MODEL = "google/gemini-3.6-flash"
FLAGS = ["--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3", "--reuse_judgments"]
JOBS = [("dpo_hh_vs_sft", "quality"), ("dpo_ultrafeedback_vs_sft", "quality"),
        ("lora_vs_base_persona", "persona_style"), ("lora_vs_base_persona", "quality")]

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nauthor date %ad%nsubject %s"], cwd=REPO_DIR)

## The judge package and model access

Only the repository's pure-Python code is needed here, so nothing is pip-installed in the benchmark environment.

In [ ]:
import glob
import json

sys.path.insert(0, f"{REPO_DIR}/src")
from pa2 import judge  # noqa: E402

import kaggle_benchmarks as kbench  # noqa: E402

available = list(kbench.llms.keys())
print(len(available), "models;", [name for name in available if "gemini" in name])
if JUDGE_MODEL not in available:
    raise RuntimeError(f"{JUDGE_MODEL} is not available in this notebook")
os.makedirs(OUT_DIR, exist_ok=True)
json.dump({"commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip(),
           "judge_model": JUDGE_MODEL, "flags": FLAGS, "jobs": JOBS}, open(f"{OUT_DIR}/run_info.json", "w"), indent=2)

## The saved pair files from Task 6

In [ ]:
found = sorted(glob.glob("/kaggle/input/**/outputs/generate/pairs/*.jsonl", recursive=True))
PAIRS = {os.path.basename(path).replace(".jsonl", ""): path for path in found}
missing = {comparison for comparison, _ in JOBS} - set(PAIRS)
if missing:
    raise RuntimeError(f"pair files not found for {missing}: attach the output of pa2-06-generate")
for name, path in PAIRS.items():
    print(f"{name:<28} {sum(1 for _ in open(path))} pairs  {path}")

## Preview: the first batch of HH pairs

These 5 verdicts are kept; the full run below resumes after them.

In [ ]:
def judge_job(comparison, criterion, extra=()):
    judge.main(["--pairs_path", PAIRS[comparison], "--output_path", f"{OUT_DIR}/{comparison}_{criterion}.jsonl",
                "--judge_model", JUDGE_MODEL, "--criterion", criterion, *FLAGS, *extra])


judge_job("dpo_hh_vs_sft", "quality", ["--limit", "5"])
for row in judge.read_jsonl(f"{OUT_DIR}/dpo_hh_vs_sft_quality.jsonl")[:5]:
    print(row["pair_id"], row["category"], "->", row["verdict"], f"({row['winner_model']})", "|", row["reason"])

## Full judging (about 136 requests, 20 to 30 minutes)

In [ ]:
for comparison, criterion in JOBS:
    print(f"\n===== {comparison} / {criterion} =====")
    judge_job(comparison, criterion)

## Results

In [ ]:
summaries = {}
for comparison, criterion in JOBS:
    summaries[f"{comparison}/{criterion}"] = json.load(open(f"{OUT_DIR}/{comparison}_{criterion}_summary.json"))
json.dump(summaries, open(f"{OUT_DIR}/all_summaries.json", "w"), indent=2)


def pct(value):
    return "  n/a" if value is None else f"{100 * value:5.1f}%"


print(f"{'judgement':<42}{'n':>5}{'valid':>6}{'win':>5}{'loss':>5}{'tie':>5}{'unk':>5}{'fail':>5}"
      f"{'win% (tie=1/2)':>16}{'win% (no ties)':>16}{'A share':>9}")
for key, s in summaries.items():
    print(f"{key:<42}{s['pairs']:>5}{s['judged_valid']:>6}{s['wins']:>5}{s['losses']:>5}{s['ties']:>5}{s['unknown']:>5}"
          f"{s['failed_batches']:>5}{pct(s['win_rate_ties_as_half']):>16}{pct(s['win_rate_without_ties']):>16}{pct(s['position_a_share']):>9}")

for key, s in summaries.items():
    print(f"\n--- {key}: {s['evaluated']} vs {s['baseline']} by category (wins / losses / ties)")
    for category, c in s["by_category"].items():
        print(f"  {category:<30} {c['wins']:>3} / {c['losses']:>3} / {c['ties']:>3}   win% no ties {pct(c['win_rate_without_ties'])}")
    length = s["length"]
    print(f"  length: evaluated longer in {length['evaluated_longer']['pairs']} pairs -> win% {pct(length['evaluated_longer']['win_rate_without_ties'])};"
          f" not longer in {length['evaluated_not_longer']['pairs']} -> win% {pct(length['evaluated_not_longer']['win_rate_without_ties'])}"
          f"; mean words {length['mean_words_evaluated']:.0f} vs {length['mean_words_baseline']:.0f}")

## Archive

In [ ]:
import zipfile


def zip_small_outputs(archive_path, root, skip_dirs=(), skip_suffixes=(".safetensors", ".jsonl.gz")):
    """Zip everything under root except checkpoints and bulky data, so the archive stays small."""
    count = 0
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for folder, dirs, files in os.walk(root):
            rel_folder = os.path.relpath(folder, root)
            dirs[:] = [d for d in dirs if os.path.normpath(os.path.join(rel_folder, d)) not in skip_dirs]
            for name in sorted(files):
                if name.endswith(skip_suffixes):
                    continue
                path = os.path.join(folder, name)
                archive.write(path, os.path.relpath(path, root))
                count += 1
    print(f"Saved {archive_path} ({count} files, {os.path.getsize(archive_path) / 1e6:.1f} MB)")


zip_small_outputs(f"{WORK_DIR}/task7_judge_outputs.zip", OUT_DIR)